# OCR improvement

## Цель
Улучшить качество распознавания номера на табличке.

## Подход
Тестирование различных вариантов:
- предобработки изображения
- выделения зоны номера
- OCR моделей
- постобработки текста

## Импорты

In [1]:
from pathlib import Path
import random
import re

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd

import easyocr
from ultralytics import YOLO

## Загрузка модели


In [2]:
PROJECT_ROOT = Path("..").resolve()
MODEL_PATH = PROJECT_ROOT / "runs" / "detect" / "train4" / "weights" / "best.pt"

model = YOLO(str(MODEL_PATH))
model

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C2f(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_s

## Функции

### Функции показа

In [3]:
def show_image(image, title="", figsize=(10, 8), cmap=None):
    plt.figure(figsize=figsize)
    plt.imshow(image, cmap=cmap)
    plt.title(title)
    plt.axis("off")
    plt.show()

In [4]:
def draw_bbox_on_image(image_rgb, bbox, title="Detection"):
    x1, y1, x2, y2 = map(int, bbox)

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(image_rgb)
    rect = patches.Rectangle(
        (x1, y1),
        x2 - x1,
        y2 - y1,
        linewidth=2,
        edgecolor="red",
        facecolor="none"
    )
    ax.add_patch(rect)
    ax.set_title(title)
    ax.axis("off")
    plt.show()

### Фукция вырезания таблички из исходного изображения

In [5]:
def crop_bbox(image_bgr, bbox):
    h, w = image_bgr.shape[:2]
    x1, y1, x2, y2 = map(int, bbox)

    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = min(w, x2)
    y2 = min(h, y2)

    return image_bgr[y1:y2, x1:x2]

### Функция нормализации ориентации

In [6]:
def normalize_plate_orientation(image_bgr):
    h, w = image_bgr.shape[:2]
    if h > w:
        image_bgr = cv2.rotate(image_bgr, cv2.ROTATE_90_CLOCKWISE)
    return image_bgr

### Функция сборки текста

In [7]:
def extract_text_from_easyocr(ocr_output):
    texts = []
    confs = []

    for item in ocr_output:
        bbox, text, conf = item
        texts.append(text)
        confs.append(conf)

    merged_text = " ".join(texts)
    mean_conf = float(np.mean(confs)) if confs else 0.0

    return merged_text, mean_conf

### Функция постобработки

In [8]:
def process_ocr_text(raw_text: str) -> str:
    text = raw_text.lower()
    text = re.sub(r"[^0-9abcde]", "", text)

    match = re.fullmatch(r"\d{1,4}[abcde]?", text)
    return match.group(0) if match else ""

### Функция нормализации номера

In [9]:
def normalize_plate_number(text: str) -> str:
    match = re.fullmatch(r"(\d{1,4})([abcde]?)", text)
    if not match:
        return ""

    digits, suffix = match.groups()
    digits = digits.zfill(4)
    return digits + suffix

### Функция выделения зоны номера

In [10]:
def extract_number_zone(plate_bgr):
    h, w = plate_bgr.shape[:2]

    x1 = int(w * 0.06)   # убираем левую подпись
    x2 = int(w * 0.84)   # убираем правую стрелку
    y1 = int(h * 0.18)   # немного сверху отступ
    y2 = int(h * 0.78)   # убираем нижнюю шкалу

    return plate_bgr[y1:y2, x1:x2]

### Функция извлечения эталона из имени файла

In [11]:
def extract_expected_number_from_filename(filename: str) -> str:
    stem = Path(filename).stem.lower()

    match = re.search(r"(\d{1,4}[abcde]?)", stem)
    if not match:
        return ""

    raw = match.group(1)
    return normalize_plate_number(raw)

## Создание OCR reader

In [12]:
reader = easyocr.Reader(['en'], gpu=False)

Using CPU. Note: This module is much faster with a GPU.


## Фиксирование выборки

In [13]:
PROJECT_ROOT = Path("..").resolve()
DATASET_DIR = PROJECT_ROOT / "dataset"
IMAGES_DIR = DATASET_DIR / "images"

image_paths = []

for split in ["train", "val"]:
    split_dir = IMAGES_DIR / split
    image_paths.extend([
        p for p in split_dir.glob("*")
        if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    ])

print("Total images:", len(image_paths))

Total images: 123


In [14]:
random.seed(42)
sample_paths = random.sample(image_paths, 20)

## Тест разных предобработок

### Вариант 1 - adaptive threshold

In [15]:
def preprocess_adaptive(gray):
    return cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        11, 2
    )

### Вариант 2 - Инверсия

In [16]:
def preprocess_invert(gray):
    _, th = cv2.threshold(gray, 130, 255, cv2.THRESH_BINARY)
    return 255 - th

### Вариант 3 - blur + threshold

In [17]:
def preprocess_blur(gray):
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, th = cv2.threshold(blur, 130, 255, cv2.THRESH_BINARY)
    return th

## Универсальный OCR запуск

In [18]:
def run_ocr_on_image(image, reader):
    result = reader.readtext(image)
    raw_text, conf = extract_text_from_easyocr(result)
    clean = process_ocr_text(raw_text)
    norm = normalize_plate_number(clean)
    return norm, conf, raw_text

## Сравнение всех вариантов

In [19]:
def test_preprocessing_variants(image_path, model, reader):
    image_bgr = cv2.imread(str(image_path))

    result = model(str(image_path), verbose=False)[0]
    if len(result.boxes) == 0:
        return None

    bbox = result.boxes.xyxy.cpu().numpy()[0]
    plate = crop_bbox(image_bgr, bbox)
    plate = normalize_plate_orientation(plate)

    zone = extract_number_zone(plate)
    gray = cv2.cvtColor(zone, cv2.COLOR_BGR2GRAY)

    variants = {
        "threshold": cv2.threshold(gray, 130, 255, cv2.THRESH_BINARY)[1],
        "adaptive": preprocess_adaptive(gray),
        "invert": preprocess_invert(gray),
        "blur": preprocess_blur(gray),
    }

    results = {}

    for name, img in variants.items():
        norm, conf, raw = run_ocr_on_image(img, reader)
        results[name] = {
            "norm": norm,
            "conf": conf,
            "raw": raw
        }

    return results

## Прогон по выборке

In [20]:
rows = []

for path in sample_paths:
    expected = extract_expected_number_from_filename(path.name)
    variants = test_preprocessing_variants(path, model, reader)

    if variants is None:
        continue

    for name, res in variants.items():
        rows.append({
            "filename": path.name,
            "method": name,
            "expected": expected,
            "predicted": res["norm"],
            "match": expected == res["norm"]
        })

df = pd.DataFrame(rows)
df

d:\projects\plate-recognition\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


,filename,method,expected,predicted,match
0,TUL082_6.jpg,threshold,0082,0082,True
1,TUL082_6.jpg,adaptive,0082,,False
2,TUL082_6.jpg,invert,0082,0824,False
3,TUL082_6.jpg,blur,0082,0082,True
4,MDZ095 _2.jpg,threshold,0095,0095,True
...,...,...,...,...,...
75,SDB274_1.jpg,blur,0274,0274,True
76,TUL051_2.jpg,threshold,0051,0005,False
77,TUL051_2.jpg,adaptive,0051,,False
78,TUL051_2.jpg,invert,0051,0051,True


## Сравнение методов

In [21]:
df.groupby("method")["match"].mean()

method
adaptive     0.00
blur         0.60
invert       0.55
threshold    0.55
Name: match, dtype: float64

## Выводы
1. Adaptive threshold — провалился
- сильно ломает цифры
- создаёт шум

2. Blur + threshold — лучший (0.60)
- сглаживает шум
- делает цифры более “OCR-friendly”

3. Invert и обычный threshold — норм, но хуже
- можно оставить как fallback

## Основной preprocessing

In [22]:
def preprocess_best(gray):
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, th = cv2.threshold(blur, 130, 255, cv2.THRESH_BINARY)
    return th

## Cортировка по X

**Проблема №1**

OCR иногда читает цифры в неправильном порядке `3 { 8 8 6 → 3886`

In [23]:
def extract_sorted_text(ocr_output):
    if not ocr_output:
        return "", 0.0

    # сортировка по X координате
    items = sorted(ocr_output, key=lambda x: x[0][0][0])

    texts = []
    confs = []

    for _, text, conf in items:
        texts.append(text)
        confs.append(conf)

    merged = " ".join(texts)
    mean_conf = float(np.mean(confs)) if confs else 0.0

    return merged, mean_conf

In [24]:
def run_ocr_on_image_v2(image, reader):
    result = reader.readtext(image)
    raw_text, conf = extract_sorted_text(result)
    clean = process_ocr_text(raw_text)
    norm = normalize_plate_number(clean)
    return norm, conf, raw_text

In [25]:
def test_preprocessing_variants_v2(image_path, model, reader):
    image_bgr = cv2.imread(str(image_path))

    result = model(str(image_path), verbose=False)[0]
    if len(result.boxes) == 0:
        return None

    bbox = result.boxes.xyxy.cpu().numpy()[0]
    plate = crop_bbox(image_bgr, bbox)
    plate = normalize_plate_orientation(plate)

    zone = extract_number_zone(plate)
    gray = cv2.cvtColor(zone, cv2.COLOR_BGR2GRAY)

    variants = {
        "threshold": cv2.threshold(gray, 130, 255, cv2.THRESH_BINARY)[1],
        "adaptive": preprocess_adaptive(gray),
        "invert": preprocess_invert(gray),
        "blur": preprocess_blur(gray),
    }

    results = {}

    for name, img in variants.items():
        norm, conf, raw = run_ocr_on_image_v2(img, reader)
        results[name] = {
            "norm": norm,
            "conf": conf,
            "raw": raw
        }

    return results

In [26]:
rows_v2 = []

for path in sample_paths:
    expected = extract_expected_number_from_filename(path.name)
    variants = test_preprocessing_variants_v2(path, model, reader)

    if variants is None:
        continue

    for name, res in variants.items():
        rows_v2.append({
            "filename": path.name,
            "method": name,
            "expected": expected,
            "predicted": res["norm"],
            "match": expected == res["norm"]
        })

df_v2 = pd.DataFrame(rows_v2)
df_v2

d:\projects\plate-recognition\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


,filename,method,expected,predicted,match
0,TUL082_6.jpg,threshold,0082,0082,True
1,TUL082_6.jpg,adaptive,0082,,False
2,TUL082_6.jpg,invert,0082,0482,False
3,TUL082_6.jpg,blur,0082,0082,True
4,MDZ095 _2.jpg,threshold,0095,0095,True
...,...,...,...,...,...
75,SDB274_1.jpg,blur,0274,0274,True
76,TUL051_2.jpg,threshold,0051,0005,False
77,TUL051_2.jpg,adaptive,0051,,False
78,TUL051_2.jpg,invert,0051,0051,True


In [27]:
print("V1:")
print(df.groupby("method")["match"].mean())

print("\nV2:")
print(df_v2.groupby("method")["match"].mean())

V1:
method
adaptive     0.00
blur         0.60
invert       0.55
threshold    0.55
Name: match, dtype: float64

V2:
method
adaptive     0.00
blur         0.60
invert       0.55
threshold    0.55
Name: match, dtype: float64


## Эксперимент 2 — сортировка OCR блоков по X

Был протестирован вариант OCR pipeline, в котором OCR-элементы дополнительно сортировались слева направо по координате X.

Результат:
- значения `match rate` не изменились по сравнению с baseline;
- улучшения качества распознавания не наблюдалось.

Вывод:
ошибки OCR в текущем pipeline связаны не столько с порядком OCR-блоков, сколько с качеством выделения зоны номера и наличием лишних символов.
Следующий этап — улучшение локализации зоны номера.

## Локализация зоны номера

In [28]:
def extract_number_zone_v1(plate_bgr):
    h, w = plate_bgr.shape[:2]
    x1 = int(w * 0.18)
    x2 = int(w * 0.84)
    y1 = int(h * 0.18)
    y2 = int(h * 0.78)
    return plate_bgr[y1:y2, x1:x2]

def extract_number_zone_v2(plate_bgr):
    h, w = plate_bgr.shape[:2]
    x1 = int(w * 0.25)
    x2 = int(w * 0.80)
    y1 = int(h * 0.18)
    y2 = int(h * 0.72)
    return plate_bgr[y1:y2, x1:x2]

def extract_number_zone_v3(plate_bgr):
    h, w = plate_bgr.shape[:2]
    x1 = int(w * 0.22)
    x2 = int(w * 0.82)
    y1 = int(h * 0.16)
    y2 = int(h * 0.68)
    return plate_bgr[y1:y2, x1:x2]

In [29]:
def preprocess_blur_only(gray):
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, th = cv2.threshold(blur, 130, 255, cv2.THRESH_BINARY)
    return th

In [30]:
def run_zone_experiment(image_path, model, reader):
    image_bgr = cv2.imread(str(image_path))

    result = model(str(image_path), verbose=False)[0]
    if len(result.boxes) == 0:
        return None

    bbox = result.boxes.xyxy.cpu().numpy()[0]
    plate = crop_bbox(image_bgr, bbox)
    plate = normalize_plate_orientation(plate)

    zone_extractors = {
        "zone_v1": extract_number_zone_v1,
        "zone_v2": extract_number_zone_v2,
        "zone_v3": extract_number_zone_v3,
    }

    results = {}

    for zone_name, zone_fn in zone_extractors.items():
        zone = zone_fn(plate)
        gray = cv2.cvtColor(zone, cv2.COLOR_BGR2GRAY)
        prep = preprocess_blur_only(gray)

        ocr_output = reader.readtext(prep)
        raw_text, conf = extract_text_from_easyocr(ocr_output)
        clean = process_ocr_text(raw_text)
        norm = normalize_plate_number(clean)

        results[zone_name] = {
            "raw": raw_text,
            "norm": norm,
            "conf": conf,
        }

    return results

In [31]:
rows_zone = []

for path in sample_paths:
    expected = extract_expected_number_from_filename(path.name)
    variants = run_zone_experiment(path, model, reader)

    if variants is None:
        continue

    for zone_name, res in variants.items():
        rows_zone.append({
            "filename": path.name,
            "zone_method": zone_name,
            "expected": expected,
            "predicted": res["norm"],
            "match": expected == res["norm"],
            "raw": res["raw"],
            "conf": res["conf"],
        })

df_zone = pd.DataFrame(rows_zone)
df_zone

d:\projects\plate-recognition\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


,filename,zone_method,expected,predicted,match,raw,conf
0,TUL082_6.jpg,zone_v1,0082,0082,True,8 2,0.723933
1,TUL082_6.jpg,zone_v2,0082,0082,True,8 2,1.000000
2,TUL082_6.jpg,zone_v3,0082,0002,False,2,1.000000
3,MDZ095 _2.jpg,zone_v1,0095,0095,True,9 5,0.669120
4,MDZ095 _2.jpg,zone_v2,0095,0095,True,9 5,0.999998
5,MDZ095 _2.jpg,zone_v3,0095,0095,True,9 5,0.999972
6,MDZ087 _8.jpg,zone_v1,0087,0087,True,8 /7 |,0.428143
7,MDZ087 _8.jpg,zone_v2,0087,0087,True,8 7,0.996834
8,MDZ087 _8.jpg,zone_v3,0087,,False,,0.000000
9,TUL855_5.jpg,zone_v1,0855,0008,False,8,0.999998


In [32]:
df_zone.groupby("zone_method")["match"].mean()

zone_method
zone_v1    0.55
zone_v2    0.60
zone_v3    0.45
Name: match, dtype: float64

## Итог экспериментов с OCR (EasyOCR)

В рамках экспериментов были протестированы:
- различные методы предобработки;
- сортировка OCR-блоков;
- несколько вариантов выделения зоны номера.

Результат:
- лучший match rate достиг ~0.60;
- дальнейшие улучшения через preprocessing и crop дают минимальный прирост.

Вывод:
качество распознавания ограничено возможностями EasyOCR в данной задаче.

Следующий этап:
сравнение с альтернативным OCR-движком (PaddleOCR).